<a href="https://colab.research.google.com/github/LiuChen-5749342/Generative-AI-and-AI-Applications/blob/main/Task_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

(check the python notebook SD4_Synthetic_Loan_Data_Generation_with_CTGAN_&_TVAE.ipynb for context and further details)

Load the file fraud_transactions.csv and create a synthetic data set of 5000 records. Evaluate the quality of the synthetic data created.


In [1]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

from scipy.stats import ks_2samp
from sklearn.neighbors import NearestNeighbors


# 1. Load the data: fraud_transactions.csv


In [3]:
# Load the dataset
file_path = "/content/fraud_transactions.csv"
df = pd.read_csv(file_path)

# Preview the data
print(df.head())

# Optional: check shape and columns
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

  trans_date_trans_time                             merchant       category  \
0         2/27/19 21:32  fraud_Langosh, Wintheiser and Hyatt    food_dining   
1         2/13/19 19:41               fraud_Dibbert and Sons  entertainment   
2         1/11/19 20:03   fraud_McDermott, Osinski and Morar           home   
3          1/20/19 9:08                   fraud_Bauch-Raynor    grocery_pos   
4          1/4/19 17:04      fraud_Reichert, Huels and Hoppe   shopping_net   

     amt gender state                            job  is_fraud  
0  83.64      F    TX             Physicist, medical         0  
1  79.13      M    PA        Secretary/administrator         0  
2  12.02      F    CA              Buyer, industrial         0  
3  84.41      M    TN  Clothing/textile technologist         0  
4   2.81      F    ME               Financial trader         0  
Shape: (6486, 8)
Columns: ['trans_date_trans_time', 'merchant', 'category', 'amt', 'gender', 'state', 'job', 'is_fraud']


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6486 entries, 0 to 6485
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  6486 non-null   object 
 1   merchant               6486 non-null   object 
 2   category               6486 non-null   object 
 3   amt                    6486 non-null   float64
 4   gender                 6486 non-null   object 
 5   state                  6486 non-null   object 
 6   job                    6486 non-null   object 
 7   is_fraud               6486 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 405.5+ KB


In [5]:
df

,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0
...,...,...,...,...,...,...,...,...
6481,1/26/19 16:26,fraud_Larson-Moen,entertainment,49.99,F,IL,Building surveyor,0
6482,2/26/19 1:27,"fraud_Kovacek, Dibbert and Ondricka",grocery_pos,84.52,F,VT,Claims inspector/assessor,0
6483,2/21/19 19:15,fraud_Lesch Ltd,shopping_pos,1.85,F,NE,"Nurse, children's",0
6484,1/23/19 18:06,fraud_Upton PLC,entertainment,171.35,F,NE,"Surveyor, minerals",0


In [6]:
df["is_fraud"].value_counts(normalize=True)

,proportion
is_fraud,
0,0.925069
1,0.074931


# 2. Process the 'is_fraud' as outcome

Our prediction / label variable is:

- `is_fraud` (1 = fraud, 0 = not fraud).

We define three groups of columns:

- **Target:** `is_fraud`
- **Categorical features:** treated as discrete category.
- **Numeric features**

Steps:
1. Drop rows with missing values (dataset is usually clean, but we are defensive).
2. Split into features `X` and target `y`.


In [7]:
target_col = "is_fraud"

# 3. Define categorical and numeric feature columns
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = df.select_dtypes(include=["number"]).columns.tolist()

# Remove target from feature lists if present
if target_col in categorical_features:
    categorical_features.remove(target_col)
if target_col in numeric_features:
    numeric_features.remove(target_col)

# 4. Split into features X and target y
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

df[[target_col]].value_counts(normalize=True)

,proportion
is_fraud,
0,0.925069
1,0.074931


## 3. Train/test split on real data

We will later:

- Train a classifier on **real data** (baseline).
- Train a classifier on **synthetic data** and test on **real data** (TSTR).

To do this, we split the real dataset into `train` and `test` with stratification on `is_fraud`.

In [8]:
real_train, real_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df[target_col],
    random_state=42
)

real_train.shape, real_test.shape

((5188, 8), (1298, 8))

## 4. Train CTGAN to generate synthetic fraud

We will use **CTGAN**, a GAN variant designed specifically for mixed‑type tabular data.

Key design choices:

- Input to CTGAN: all feature columns **plus** the outcome `is_fraud`.
- `discrete_columns` parameter: includes all categorical fields and integer count features, including the binary outcome.

This lets us later **condition** on `is_fraud` if we want to oversample non fraud.


In [9]:
from ctgan import CTGAN
# Prepare CTGAN training data:
train_ctgan_df = df.copy()

#    Build discrete_columns for CTGAN
#    include:
#    - all categorical features
#    - integer/count/binary columns
#    - the binary target is_fraud
discrete_columns = list(categorical_features)

for col in train_ctgan_df.columns:
    if col in discrete_columns:
        continue

    # Treat integer-like columns as discrete
    if pd.api.types.is_integer_dtype(train_ctgan_df[col]):
        discrete_columns.append(col)
    # Also treat binary float columns as discrete if applicable
    elif pd.api.types.is_float_dtype(train_ctgan_df[col]):
        unique_vals = train_ctgan_df[col].dropna().unique()
        if len(unique_vals) <= 2 and set(unique_vals).issubset({0.0, 1.0}):
            discrete_columns.append(col)

# Ensure target is included
if target_col not in discrete_columns:
    discrete_columns.append(target_col)

# Optional: remove duplicates while preserving order
discrete_columns = list(dict.fromkeys(discrete_columns))

# Train CTGAN
ctgan = CTGAN(
    epochs=300,          # adjust as needed
    batch_size=500,      # adjust based on dataset size / memory
    verbose=True
)

ctgan.fit(train_ctgan_df, discrete_columns=discrete_columns)

# Optional: inspect setup
print("CTGAN training data shape:", train_ctgan_df.shape)
print("Discrete columns:", discrete_columns)

Gen. (+00.00) | Discrim. (+00.00):   0%|          | 0/300 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Gen. (+00.73) | Discrim. (-00.03): 100%|██████████| 300/300 [16:12<00:00,  3.24s/it]

CTGAN training data shape: (6486, 8)
Discrete columns: ['trans_date_trans_time', 'merchant', 'category', 'gender', 'state', 'job', 'is_fraud']


## 5. Generate synthetic data

Let's generate 5,000 synthetic records.

We will inspect:

- Schema (column names, dtypes).
- Class balance for `is_fraud` in synthetic data vs. real data.

In [10]:
n_synth = 5000
synthetic = ctgan.sample(n_synth)
synthetic.head()

,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,1/20/19 20:01,fraud_Schoen Ltd,gas_transport,162.169230,F,NE,Information systems manager,0
1,2/24/19 7:47,fraud_Romaguera and Sons,misc_net,14.119913,F,CO,Orthoptist,0
2,2/26/19 22:42,fraud_Kerluke Inc,grocery_pos,56.736034,F,NE,Chief Strategy Officer,0
3,1/27/19 18:37,fraud_Brown Inc,food_dining,115.996534,M,NM,Physiotherapist,0
4,1/9/19 13:48,fraud_Champlin and Sons,misc_net,18.668376,F,NY,"Engineer, production",0


In [11]:
synthetic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   trans_date_trans_time  5000 non-null   object 
 1   merchant               5000 non-null   object 
 2   category               5000 non-null   object 
 3   amt                    5000 non-null   float64
 4   gender                 5000 non-null   object 
 5   state                  5000 non-null   object 
 6   job                    5000 non-null   object 
 7   is_fraud               5000 non-null   int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 312.6+ KB


In [12]:
print("Real outcome distribution:")
print(df[target_col].value_counts(normalize=True))

print("\nSynthetic outcome distribution:")
print(synthetic[target_col].value_counts(normalize=True))

Real outcome distribution:
is_fraud
0    0.925069
1    0.074931
Name: proportion, dtype: float64

Synthetic outcome distribution:
is_fraud
0    0.9008
1    0.0992
Name: proportion, dtype: float64


## 6. Is the Synthetic Data Trustworthy?

We will evaluate the generated synthetic data set along the three pillars of quality.

1. **Fidelity**: Does the synthetic data statistically resemble the real data?
2. **Utility**: Can a machine learning algorithm trained on synthetic data perform well on real data?
3. **Privacy**: Can we guarantee that the synthetic data does not expose sentitive information from the real data?

### 6.1 Fidelity: do synthetic and real loans look statistically similar?

We evaluate fidelity for numeric columns based on individual columns and pairwise correlations

#### 6.1.1 Univariate Fidelity:

- For each numeric column, run a **Kolmogorov–Smirnov (KS) test** comparing real vs synthetic samples.
- KS statistic near 0 ⇒ distributions are similar.
- Higher KS ⇒ synthetic deviates from real.

We do this feature by feature.


In [15]:
real = train_ctgan_df
syn = synthetic

ks_results = {}

for col in discrete_columns:
    r = real[col].sample(min(2000, len(real)), random_state=42)
    s = syn[col].sample(min(2000, len(syn)), random_state=42)
    stat, pval = ks_2samp(r, s)
    ks_results[col] = {"ks_stat": stat, "p_value": pval}

ks_df = pd.DataFrame(ks_results).T.sort_values("ks_stat")
ks_df

,ks_stat,p_value
is_fraud,0.0125,9.976509e-01
job,0.0220,7.185143e-01
trans_date_trans_time,0.0260,5.085995e-01
merchant,0.0370,1.293722e-01
state,0.0375,1.200888e-01
category,0.0490,1.641475e-02
gender,0.1420,5.393681e-18


###' 6.1.2 Correlation Structure

We now compare **pairwise correlations** between numerical features in real vs synthetic data.

- Compute correlation matrices for real and synthetic numeric features.
- Look at absolute differences between them.

In [21]:
real_corr = real[numeric_features].corr()
syn_corr = syn[numeric_features].corr()

corr_diff = abs(real_corr - syn_corr)
corr_diff

,amt
amt,0.0


In [22]:
# Average absolute difference in correlation per feature
corr_diff.mean().sort_values(ascending=False)

,0
amt,0.0


Explaination: Since the numeric_feature only inculdes amt feature, it has the corr related to itself

### 6.2 Utility: can synthetic data train a useful default model?

We evaluate **utility** using the TSTR protocol:

1. Fit a classifier on **synthetic** data.
2. Test on **held‑out real** data.
3. Compare performance with a classifier trained on **real** data (upper bound).

Metrics:

- ROC AUC
- F1‑score (for imbalanced classification)

#### 6.2.1 Create encoders/scaler on REAL training data

We will:

- Fit **OneHotEncoder** and **StandardScaler** only on the **real training** subset.
- Apply exactly the same transformations to synthetic data and real test data.

In [24]:
# Use the existing lists: categorical_features and numeric_features
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

# Fit on REAL TRAINING DATA ONLY using the correct variable names
ohe.fit(real_train[categorical_features])
scaler.fit(real_train[numeric_features])

def preprocess_for_model(df_subset, ohe, scaler, cat_cols, num_cols, target_col):
    df_subset = df_subset.copy()

    X_cat = df_subset[cat_cols]
    X_num = df_subset[num_cols]
    y_out = df_subset[target_col].astype(int)

    X_cat_enc = ohe.transform(X_cat)
    X_num_scaled = scaler.transform(X_num)

    X_all = np.hstack([X_cat_enc, X_num_scaled])
    return X_all, y_out

#### 6.2.2 Baseline: Train on REAL, Test on REAL

This is our **upper bound** for performance.

In [32]:
# 1. Re-initialize and fit the preprocessors correctly
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
scaler = StandardScaler()

ohe.fit(real_train[categorical_features])
scaler.fit(real_train[numeric_features])

# 2. Transform the real data
X_real_train, y_real_train = preprocess_for_model(real_train, ohe, scaler, categorical_features, numeric_features, target_col)
X_real_test, y_real_test = preprocess_for_model(real_test, ohe, scaler, categorical_features, numeric_features, target_col)

# 3. Train the Random Forest Baseline
rf_real = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_real.fit(X_real_train, y_real_train)

# 4. Evaluate
y_proba_real = rf_real.predict_proba(X_real_test)[:, 1]
y_pred_real = (y_proba_real >= 0.5).astype(int)

auc_real = roc_auc_score(y_real_test, y_proba_real)
f1_real = f1_score(y_real_test, y_pred_real)

print(f"Real Data Baseline - AUC: {auc_real:.4f}, F1: {f1_real:.4f}")
(auc_real, f1_real)

Real Data Baseline - AUC: 0.9763, F1: 0.8276


(np.float64(0.9763470303956324), 0.8275862068965517)

#### 6.2.3 TSTR: Train on SYNTHETIC, Test on REAL

We now:

- Use our `synthetic` dataframe as training data.
- Evaluate on the **same real test set** as above.

This tells us how good models trained purely on synthetic data are for predicting `not.fully.paid` on real loans.

In [33]:
# Pass the required arguments to preprocess_for_model
X_syn_train, y_syn_train = preprocess_for_model(
    synthetic,
    ohe,
    scaler,
    categorical_features,
    numeric_features,
    target_col
)

rf_syn = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf_syn.fit(X_syn_train, y_syn_train)

# Evaluate on the same real test set
y_proba_syn = rf_syn.predict_proba(X_real_test)[:, 1]
y_pred_syn = (y_proba_syn >= 0.5).astype(int)

auc_syn = roc_auc_score(y_real_test, y_proba_syn)
f1_syn = f1_score(y_real_test, y_pred_syn)

print(f"TSTR (Synthetic Train, Real Test) - AUC: {auc_syn:.4f}, F1: {f1_syn:.4f}")
(auc_syn, f1_syn)

TSTR (Synthetic Train, Real Test) - AUC: 0.7288, F1: 0.2901


(np.float64(0.7288170510828605), 0.2900763358778626)

In [35]:
pd.DataFrame({"AUC": [auc_real, auc_syn], "F1": [f1_real, f1_syn]},
             index = ['Train REAL, Test REAL', "Train SYN, Test REAL"])

,AUC,F1
"Train REAL, Test REAL",0.976347,0.827586
"Train SYN, Test REAL",0.728817,0.290076


### 6.3 Privacy: Approximate Memorization Check

CTGAN can overfit and memorize real rows, which is a privacy risk.

A simple heuristic:
- Take numeric features from synthetic data.
- For each synthetic point, compute the distance to the **nearest real point**.
- If many synthetic points are at extremely small distance, it may indicate memorization.

This is not a formal privacy guarantee, but a useful metric.

In [36]:
# Sample down for speed
real_num = real[numeric_features].sample(2000, random_state=42)
syn_num = syn[numeric_features].sample(2000, random_state=42)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_num)

distances, indices = nn.kneighbors(syn_num)
distances = distances.flatten()

pd.Series(distances).describe()

,0
count,2000.000000
mean,0.261695
std,1.183694
min,0.000002
25%,0.010212
50%,0.028770
75%,0.078813
max,18.855169
